# Enhanced Semiconductor Defect Detection - 4 Categories
## MobileNetV3-Small for Edge-AI Deployment

**Categories**: LER, Bridge, Particles, Other  
**Dataset**: 2400 synthetic images (1680 train, 360 val, 360 test)  
**Model**: MobileNetV3-Small (optimized for edge devices)

## 1. Setup & Installation

In [ ]:
# Install required packages
!pip install tensorflow==2.15.0 numpy matplotlib pillow scikit-learn seaborn -q

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# GPU configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU Available: {len(gpus)} device(s)")
    except RuntimeError as e:
        print(e)
else:
    print("⚠️  No GPU detected, using CPU")

print(f"✅ TensorFlow version: {tf.__version__}")

## 2. Configuration

In [ ]:
# Dataset paths
DATASET_ROOT = r"c:\Mugi\Projects\IISE\AI model\Dataset"
TRAIN_DIR = os.path.join(DATASET_ROOT)
OUTPUT_DIR = r"c:\Mugi\Projects\IISE\AI model\outputs"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model configuration
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
CLASS_NAMES = ['Bridge', 'LER', 'Other', 'Particles']  # Alphabetical order

# Training configuration
INITIAL_EPOCHS = 5  # Phase 1: Train classifier head
FINE_TUNE_EPOCHS = 20  # Phase 2: Fine-tune entire model
INITIAL_LR = 0.001  # Higher learning rate for faster convergence
FINE_TUNE_LR = 0.0001

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Dataset root: {DATASET_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Initial LR: {INITIAL_LR}")
print(f"Fine-tune LR: {FINE_TUNE_LR}")
print("="*60)

## 3. Data Loading & Augmentation

In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=90,  # 0, 90, 180, 270 degrees
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='reflect'
)

# Validation and test data (no augmentation, only rescaling)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Load training data
train_generator = train_datagen.flow_from_directory(
    DATASET_ROOT,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset=None,
    shuffle=True,
    seed=42,
    color_mode='grayscale',
    interpolation='bilinear'
)

# Manually create train/val/test generators from subdirectories
def create_generators_from_subdirs(root_dir, class_names, img_size, batch_size):
    """Create separate generators for train/val/test from subdirectory structure."""
    
    # Training data
    train_gen = train_datagen.flow_from_directory(
        root_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='categorical',
        classes=class_names,
        shuffle=True,
        seed=42,
        color_mode='grayscale'
    )
    
    # Validation data
    val_gen = val_test_datagen.flow_from_directory(
        root_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='categorical',
        classes=class_names,
        shuffle=False,
        color_mode='grayscale'
    )
    
    # Test data
    test_gen = val_test_datagen.flow_from_directory(
        root_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='categorical',
        classes=class_names,
        shuffle=False,
        color_mode='grayscale'
    )
    
    return train_gen, val_gen, test_gen

print("\n" + "="*60)
print("DATA LOADING SUMMARY")
print("="*60)
print(f"Found {train_generator.samples} total images")
print(f"Class indices: {train_generator.class_indices}")
print("="*60)

In [ ]:
# Create custom data loaders for train/val/test splits
def load_split_data(dataset_root, class_names, split='train', img_size=IMG_SIZE, batch_size=BATCH_SIZE):
    """Load data from specific split (train/val/test) across all categories."""
    
    images = []
    labels = []
    
    for class_idx, class_name in enumerate(class_names):
        class_split_dir = Path(dataset_root) / class_name / split
        if not class_split_dir.exists():
            print(f"⚠️  Warning: {class_split_dir} not found")
            continue
        
        image_files = list(class_split_dir.glob("*.png")) + list(class_split_dir.glob("*.jpg"))
        
        for img_path in image_files:
            try:
                img = keras.preprocessing.image.load_img(
                    str(img_path), 
                    target_size=img_size,
                    color_mode='grayscale'
                )
                img_array = keras.preprocessing.image.img_to_array(img)
                images.append(img_array)
                labels.append(class_idx)
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
    
    images = np.array(images) / 255.0  # Normalize
    labels = keras.utils.to_categorical(labels, num_classes=len(class_names))
    
    print(f"✅ Loaded {split}: {len(images)} images")
    return images, labels

# Load all splits
print("\nLoading dataset splits...")
X_train, y_train = load_split_data(DATASET_ROOT, CLASS_NAMES, 'train')
X_val, y_val = load_split_data(DATASET_ROOT, CLASS_NAMES, 'val')
X_test, y_test = load_split_data(DATASET_ROOT, CLASS_NAMES, 'test')

print("\n" + "="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Training:   {X_train.shape[0]} images")
print(f"Validation: {X_val.shape[0]} images")
print(f"Test:       {X_test.shape[0]} images")
print(f"Total:      {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]} images")
print(f"Image shape: {X_train.shape[1:]}")
print("="*60)

## 4. Visualize Sample Images

In [ ]:
# Visualize samples from each class
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle('Sample Images from Each Category', fontsize=16, fontweight='bold')

for class_idx, class_name in enumerate(CLASS_NAMES):
    # Get indices for this class
    class_indices = np.where(np.argmax(y_train, axis=1) == class_idx)[0]
    
    # Select 4 random samples
    sample_indices = np.random.choice(class_indices, 4, replace=False)
    
    for i, idx in enumerate(sample_indices):
        ax = axes[class_idx, i]
        ax.imshow(X_train[idx].squeeze(), cmap='gray')
        ax.axis('off')
        if i == 0:
            ax.set_title(class_name, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✅ Sample visualization saved")

## 5. Build Model Architecture

In [ ]:
def build_model(num_classes=NUM_CLASSES, img_size=IMG_SIZE):
    """Build MobileNetV3-Small model for grayscale defect classification."""
    
    # Input layer for grayscale images
    inputs = layers.Input(shape=(*img_size, 1), name='input_layer')
    
    # Convert grayscale to RGB (MobileNetV3 expects 3 channels)
    x = layers.Concatenate()([inputs, inputs, inputs])
    
    # Load MobileNetV3-Small pretrained on ImageNet
    base_model = MobileNetV3Small(
        input_shape=(*img_size, 3),
        include_top=False,
        weights='imagenet',
        pooling='avg'
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Extract features
    x = base_model(x, training=False)
    
    # Classification head
    x = layers.Dropout(0.3, name='dropout_1')(x)
    x = layers.Dense(256, activation='relu', name='dense_1')(x)
    x = layers.BatchNormalization(name='bn_1')(x)
    x = layers.Dropout(0.2, name='dropout_2')(x)
    x = layers.Dense(128, activation='relu', name='dense_2')(x)
    x = layers.BatchNormalization(name='bn_2')(x)
    
    # Output layer
    outputs = layers.Dense(num_classes, activation='softmax', name='output_layer')(x)
    
    # Create model
    model = models.Model(inputs=inputs, outputs=outputs, name='DefectDetector_MobileNetV3')
    
    return model, base_model

# Build the model
model, base_model = build_model()

print("="*60)
print("MODEL ARCHITECTURE")
print("="*60)
print(f"Total parameters: {model.count_params():,}")
print(f"Trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")
print(f"Non-trainable parameters: {sum([tf.size(w).numpy() for w in model.non_trainable_weights]):,}")
print("="*60)

model.summary()

## 6. Phase 1: Train Classifier Head

In [ ]:
# Compile model for Phase 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

# Callbacks for Phase 1
callbacks_phase1 = [
    ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'phase1_best.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

print("\n" + "="*60)
print("PHASE 1: TRAINING CLASSIFIER HEAD")
print("="*60)
print(f"Epochs: {INITIAL_EPOCHS}")
print(f"Learning rate: {INITIAL_LR}")
print(f"Base model frozen: {not base_model.trainable}")
print("="*60 + "\n")

# Train Phase 1
history_phase1 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=INITIAL_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_phase1,
    verbose=1
)

print("\n✅ Phase 1 training complete!")

## 7. Phase 2: Fine-Tune Entire Model

In [ ]:
# Unfreeze base model for fine-tuning
base_model.trainable = True

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

# Callbacks for Phase 2
callbacks_phase2 = [
    ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'best_model.h5'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    TensorBoard(
        log_dir=os.path.join(OUTPUT_DIR, 'logs'),
        histogram_freq=1
    )
]

print("\n" + "="*60)
print("PHASE 2: FINE-TUNING ENTIRE MODEL")
print("="*60)
print(f"Epochs: {FINE_TUNE_EPOCHS}")
print(f"Learning rate: {FINE_TUNE_LR}")
print(f"Base model frozen: {not base_model.trainable}")
print(f"Trainable parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")
print("="*60 + "\n")

# Train Phase 2
history_phase2 = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=FINE_TUNE_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks_phase2,
    verbose=1
)

print("\n✅ Phase 2 fine-tuning complete!")

# Save final model
model.save(os.path.join(OUTPUT_DIR, 'final_model.h5'))
print(f"✅ Final model saved to: {os.path.join(OUTPUT_DIR, 'final_model.h5')}")

## 8. Training History Visualization

In [ ]:
# Combine histories
def plot_training_history(history1, history2, output_dir):
    """Plot combined training history from both phases."""
    
    # Combine histories
    combined_history = {
        'accuracy': history1.history['accuracy'] + history2.history['accuracy'],
        'val_accuracy': history1.history['val_accuracy'] + history2.history['val_accuracy'],
        'loss': history1.history['loss'] + history2.history['loss'],
        'val_loss': history1.history['val_loss'] + history2.history['val_loss']
    }
    
    epochs_phase1 = len(history1.history['accuracy'])
    total_epochs = len(combined_history['accuracy'])
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Accuracy plot
    ax1.plot(combined_history['accuracy'], label='Training Accuracy', linewidth=2)
    ax1.plot(combined_history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    ax1.axvline(x=epochs_phase1-0.5, color='red', linestyle='--', label='Fine-tuning starts', alpha=0.7)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
    ax1.legend(loc='lower right')
    ax1.grid(True, alpha=0.3)
    
    # Loss plot
    ax2.plot(combined_history['loss'], label='Training Loss', linewidth=2)
    ax2.plot(combined_history['val_loss'], label='Validation Loss', linewidth=2)
    ax2.axvline(x=epochs_phase1-0.5, color='red', linestyle='--', label='Fine-tuning starts', alpha=0.7)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_history.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n📊 Final Training Accuracy: {combined_history['accuracy'][-1]:.4f}")
    print(f"📊 Final Validation Accuracy: {combined_history['val_accuracy'][-1]:.4f}")
    print(f"📊 Best Validation Accuracy: {max(combined_history['val_accuracy']):.4f}")

plot_training_history(history_phase1, history_phase2, OUTPUT_DIR)
print("✅ Training history visualization saved")

## 9. Model Evaluation on Test Set

In [ ]:
# Load best model
best_model = keras.models.load_model(os.path.join(OUTPUT_DIR, 'best_model.h5'))

# Evaluate on test set
print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

test_loss, test_accuracy, test_precision, test_recall = best_model.evaluate(
    X_test, y_test, 
    batch_size=BATCH_SIZE,
    verbose=1
)

print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"Test Loss:      {test_loss:.4f}")
print(f"Test Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Test Precision: {test_precision:.4f} ({test_precision*100:.2f}%)")
print(f"Test Recall:    {test_recall:.4f} ({test_recall*100:.2f}%)")
print(f"Test F1-Score:  {2 * (test_precision * test_recall) / (test_precision + test_recall):.4f}")
print("="*60)

## 10. Enhanced Confusion Matrix

In [ ]:
# Get predictions
y_pred_probs = best_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Enhanced confusion matrix visualization
def plot_enhanced_confusion_matrix(cm, class_names, output_dir):
    """Plot beautiful, annotated confusion matrix."""
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Calculate percentages
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Create annotations with counts and percentages
    annotations = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            count = cm[i, j]
            percent = cm_percent[i, j]
            annotations[i, j] = f"{count}\n({percent:.1f}%)"
    
    # Plot heatmap
    sns.heatmap(
        cm, 
        annot=annotations, 
        fmt='', 
        cmap='Blues', 
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Number of Predictions'},
        linewidths=1,
        linecolor='gray',
        ax=ax
    )
    
    ax.set_xlabel('Predicted Label', fontsize=13, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=13, fontweight='bold')
    ax.set_title('Confusion Matrix - Defect Classification', fontsize=15, fontweight='bold', pad=20)
    
    # Rotate labels
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    plt.setp(ax.get_yticklabels(), rotation=0)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'confusion_matrix.png'), dpi=200, bbox_inches='tight')
    plt.show()
    
    print("✅ Enhanced confusion matrix saved")

plot_enhanced_confusion_matrix(cm, CLASS_NAMES, OUTPUT_DIR)

## 11. Detailed Classification Report

In [ ]:
# Classification report
print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
print("="*60)

# Per-class accuracy
print("\nPER-CLASS ACCURACY:")
for i, class_name in enumerate(CLASS_NAMES):
    class_mask = (y_true == i)
    class_acc = accuracy_score(y_true[class_mask], y_pred[class_mask])
    print(f"  {class_name:12s}: {class_acc:.4f} ({class_acc*100:.2f}%)")

# Save classification report
report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
with open(os.path.join(OUTPUT_DIR, 'classification_report.json'), 'w') as f:
    json.dump(report_dict, f, indent=2)

print("\n✅ Classification report saved")

## 12. Prediction Confidence Analysis

In [ ]:
# Analyze prediction confidence
max_confidences = np.max(y_pred_probs, axis=1)
correct_predictions = (y_pred == y_true)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence distribution
axes[0].hist(max_confidences, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=0.7, color='red', linestyle='--', label='Threshold (0.7)', linewidth=2)
axes[0].set_xlabel('Prediction Confidence', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Prediction Confidence Distribution', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Confidence vs Correctness
axes[1].scatter(
    max_confidences[correct_predictions], 
    np.ones(sum(correct_predictions)), 
    alpha=0.5, 
    label='Correct', 
    color='green',
    s=30
)
axes[1].scatter(
    max_confidences[~correct_predictions], 
    np.zeros(sum(~correct_predictions)), 
    alpha=0.5, 
    label='Incorrect', 
    color='red',
    s=30
)
axes[1].axvline(x=0.7, color='orange', linestyle='--', label='Threshold (0.7)', linewidth=2)
axes[1].set_xlabel('Prediction Confidence', fontsize=12)
axes[1].set_ylabel('Prediction Correctness', fontsize=12)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Incorrect', 'Correct'])
axes[1].set_title('Confidence vs Correctness', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confidence_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("CONFIDENCE STATISTICS")
print("="*60)
print(f"Mean confidence: {np.mean(max_confidences):.4f}")
print(f"Median confidence: {np.median(max_confidences):.4f}")
print(f"Min confidence: {np.min(max_confidences):.4f}")
print(f"Max confidence: {np.max(max_confidences):.4f}")
print(f"\nPredictions with confidence > 0.7: {sum(max_confidences > 0.7)} ({sum(max_confidences > 0.7)/len(max_confidences)*100:.1f}%)")
print(f"Predictions with confidence > 0.9: {sum(max_confidences > 0.9)} ({sum(max_confidences > 0.9)/len(max_confidences)*100:.1f}%)")
print("="*60)

print("\n✅ Confidence analysis saved")

## 13. Single Image Prediction Function

In [ ]:
def predict_single_image(model, img_path, class_names, img_size=IMG_SIZE, confidence_threshold=0.7):
    """Predict defect class for a single image with confidence."""
    
    # Load and preprocess image
    img = keras.preprocessing.image.load_img(
        img_path, 
        target_size=img_size,
        color_mode='grayscale'
    )
    img_array = keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    
    # Predict
    predictions = model.predict(img_array, verbose=0)[0]
    predicted_class_idx = np.argmax(predictions)
    confidence = predictions[predicted_class_idx]
    
    # Get top 3 predictions
    top3_indices = np.argsort(predictions)[::-1][:3]
    top3_predictions = [(class_names[i], predictions[i]) for i in top3_indices]
    
    # Determine final prediction
    if confidence >= confidence_threshold:
        final_class = class_names[predicted_class_idx]
        status = "CONFIDENT"
    else:
        final_class = class_names[predicted_class_idx]  # Still show best guess
        status = "LOW_CONFIDENCE"
    
    return {
        'predicted_class': final_class,
        'confidence': float(confidence),
        'status': status,
        'top3_predictions': top3_predictions,
        'all_probabilities': {class_names[i]: float(predictions[i]) for i in range(len(class_names))}
    }

# Test on a random image
test_idx = np.random.randint(0, len(X_test))
test_img_path = Path(DATASET_ROOT) / CLASS_NAMES[y_true[test_idx]] / 'test'
test_img_files = list(test_img_path.glob("*.png"))

if len(test_img_files) > 0:
    sample_img = str(test_img_files[0])
    result = predict_single_image(best_model, sample_img, CLASS_NAMES)
    
    print("\n" + "="*60)
    print("SAMPLE PREDICTION")
    print("="*60)
    print(f"Image: {Path(sample_img).name}")
    print(f"Predicted Class: {result['predicted_class']}")
    print(f"Confidence: {result['confidence']:.4f} ({result['confidence']*100:.2f}%)")
    print(f"Status: {result['status']}")
    print(f"\nTop 3 Predictions:")
    for i, (cls, prob) in enumerate(result['top3_predictions'], 1):
        print(f"  {i}. {cls:12s}: {prob:.4f} ({prob*100:.2f}%)")
    print("="*60)
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Show image
    img_display = keras.preprocessing.image.load_img(sample_img, color_mode='grayscale')
    ax1.imshow(img_display, cmap='gray')
    ax1.axis('off')
    ax1.set_title(f"Test Image\nTrue: {CLASS_NAMES[y_true[test_idx]]}", fontsize=12, fontweight='bold')
    
    # Show probabilities
    classes = list(result['all_probabilities'].keys())
    probs = list(result['all_probabilities'].values())
    colors = ['green' if c == result['predicted_class'] else 'gray' for c in classes]
    
    ax2.barh(classes, probs, color=colors, alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Probability', fontsize=12)
    ax2.set_title(f"Prediction: {result['predicted_class']}\nConfidence: {result['confidence']:.2%}", 
                  fontsize=12, fontweight='bold')
    ax2.set_xlim([0, 1])
    ax2.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'sample_prediction.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Sample prediction visualization saved")

## 14. Export Model for Edge Deployment

In [ ]:
# Convert to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save TFLite model
tflite_path = os.path.join(OUTPUT_DIR, 'defect_detector.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

# Get model sizes
h5_size = os.path.getsize(os.path.join(OUTPUT_DIR, 'best_model.h5')) / (1024 * 1024)
tflite_size = os.path.getsize(tflite_path) / (1024 * 1024)

print("\n" + "="*60)
print("MODEL EXPORT SUMMARY")
print("="*60)
print(f"Keras model (.h5):     {h5_size:.2f} MB")
print(f"TFLite model (.tflite): {tflite_size:.2f} MB")
print(f"Size reduction:         {(1 - tflite_size/h5_size)*100:.1f}%")
print(f"\nTFLite model saved to: {tflite_path}")
print("="*60)

print("\n✅ Model exported for edge deployment!")

## 15. Final Summary

In [ ]:
# Create summary report
summary = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': {
        'total_images': X_train.shape[0] + X_val.shape[0] + X_test.shape[0],
        'train': X_train.shape[0],
        'val': X_val.shape[0],
        'test': X_test.shape[0],
        'classes': CLASS_NAMES,
        'num_classes': NUM_CLASSES
    },
    'model': {
        'architecture': 'MobileNetV3-Small',
        'total_parameters': int(model.count_params()),
        'input_shape': list(IMG_SIZE) + [1],
        'output_classes': NUM_CLASSES
    },
    'training': {
        'phase1_epochs': INITIAL_EPOCHS,
        'phase2_epochs': FINE_TUNE_EPOCHS,
        'initial_lr': INITIAL_LR,
        'fine_tune_lr': FINE_TUNE_LR,
        'batch_size': BATCH_SIZE
    },
    'performance': {
        'test_accuracy': float(test_accuracy),
        'test_precision': float(test_precision),
        'test_recall': float(test_recall),
        'test_f1': float(2 * (test_precision * test_recall) / (test_precision + test_recall))
    },
    'model_files': {
        'keras_model': 'best_model.h5',
        'tflite_model': 'defect_detector.tflite',
        'keras_size_mb': float(h5_size),
        'tflite_size_mb': float(tflite_size)
    }
}

# Save summary
with open(os.path.join(OUTPUT_DIR, 'training_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*70)
print("TRAINING COMPLETE - FINAL SUMMARY")
print("="*70)
print(f"\n📊 Dataset: {summary['dataset']['total_images']} images ({NUM_CLASSES} classes)")
print(f"🏗️  Model: {summary['model']['architecture']} ({summary['model']['total_parameters']:,} parameters)")
print(f"\n🎯 Test Performance:")
print(f"   Accuracy:  {summary['performance']['test_accuracy']:.4f} ({summary['performance']['test_accuracy']*100:.2f}%)")
print(f"   Precision: {summary['performance']['test_precision']:.4f} ({summary['performance']['test_precision']*100:.2f}%)")
print(f"   Recall:    {summary['performance']['test_recall']:.4f} ({summary['performance']['test_recall']*100:.2f}%)")
print(f"   F1-Score:  {summary['performance']['test_f1']:.4f} ({summary['performance']['test_f1']*100:.2f}%)")
print(f"\n💾 Model Files:")
print(f"   Keras (.h5):     {summary['model_files']['keras_size_mb']:.2f} MB")
print(f"   TFLite (.tflite): {summary['model_files']['tflite_size_mb']:.2f} MB")
print(f"\n📁 Output directory: {OUTPUT_DIR}")
print("="*70)

print("\n✅ All files saved successfully!")
print("\n🚀 Model ready for edge deployment!")